<a href="https://colab.research.google.com/github/dewmindi/autoGrader/blob/dewmindi-do-4/ModelReady_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
from google.colab import files
import csv
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import Counter
import re

In [43]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [44]:
uploaded = files.upload()


data = []
with open('TestSet.csv', mode='r') as file:
    csvFile = csv.reader(file)
    header = next(csvFile)
    for lines in csvFile:
        data.append(lines)

Saving TestSet.csv to TestSet.csv


In [45]:
answer_index =1
real_answer_index = 0
rating_index = 2

def test_text(text):
    test_text = text.replace(",", "").replace(".", "").replace(":", "").strip()
    return test_text

def clean_text(text):
    """Clean text by ensuring proper spacing after punctuation."""
    # Add a space after punctuation if not followed by a space
    text = re.sub(r'(?<=[.,;!?])(?=[^\s])', r' ', text)
    return text.strip()

def count_words(text):
    """Count the total number of words in the text."""
    cleaned_text = text.replace(",", "").replace(".", "").replace(":", "").strip()
    return len(word_tokenize(cleaned_text))

def count_unique_words(text):
    """Count the number of unique words in the text."""
    cleaned_text = text.replace(",", "").replace(".", "").replace(":", "").strip()
    return len(set(word_tokenize(cleaned_text)))

def count_stop_words(text):
    """Count the number of stop words in the text."""
    stop_words_set = set(stopwords.words('english'))
    cleaned_text = text.replace(",", "").replace(".", "").replace(":", "").strip()
    return sum(1 for word in word_tokenize(cleaned_text) if word.lower() in stop_words_set)

def count_non_stop_words(text):
    """Count the number of non-stop words in the text."""
    return count_words(text) - count_stop_words(text)

def count_sentences(text):
    """Count the number of sentences in the text."""
    cleaned_text = clean_text(text)
    return len(sent_tokenize(cleaned_text))

def count_long_words(text, length=5):
    """Count the number of words longer than a given length."""
    cleaned_text = clean_text(text)
    return sum(1 for word in word_tokenize(cleaned_text) if len(word) > length)

def unique_words_with_frequency(test_text):
    tokenized = word_tokenize(test_text)
    return dict(Counter(tokenized))

# #### Grammar checking####
# def check_grammar_and_score(text):
#     """Check grammar and return the number of issues and a normalized grammar score."""
#     # Initialize the LanguageTool instance
#     tool = language_tool_python.LanguageTool('en-US')

#     # Check for grammar issues
#     matches = tool.check(text)
#     num_issues = len(matches)
#     grammar_issue_details = [
#             {
#                 'message': match.message,
#                 'context': match.context,
#                 'suggestions': match.replacements,
#                 'offset': match.offset,
#             }
#             for match in matches
#         ]

#     # Calculate total words in the text
#     total_words = len(word_tokenize(text))

#     # Normalize the grammar score
#     score = max(0, 100 - (num_issues / total_words) * 100) if total_words > 0 else 0

#     return num_issues, round(score, 2), grammar_issue_details

In [46]:
processed_data = []
header = ['Real Answer','Answer', 'Words', 'Unique_Words', 'Stop_Words',
          'Non_Stop_Words', 'Sentences', 'Long_Words', 'Word_Frequencies', 'Rating']
processed_data.append(header)

for row in data:  # Process only the first 100 rows for testing
    Real_Answer = row[real_answer_index].lower().strip()
    Answer = row[answer_index].lower().strip()
    Words = count_words(Answer)
    no_of_unique_words = count_unique_words(Answer)
    no_of_stop_words = count_stop_words(Answer)
    no_of_non_stop_words = count_non_stop_words(Answer)
    no_of_sentences = count_sentences(Answer)
    no_of_long_words = count_long_words(Answer)
    Word_Frequencies = unique_words_with_frequency(Answer)
    Rating = row[rating_index]

    # grammar_issues, grammar_score, grammar_issue_details = check_grammar_and_score(Answer)


    processed_data.append([Real_Answer,Answer, Words, no_of_unique_words, no_of_stop_words,
                           no_of_non_stop_words, no_of_sentences, no_of_long_words, Word_Frequencies, Rating])

In [47]:
import pandas as pd

for row in processed_data[1:]:
    row[-1] = str(row[-1])

output_file = "processed_TestSet.csv"
df = pd.DataFrame(processed_data[1:], columns=processed_data[0])
df.to_csv(output_file, index=False)

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Cosine Similarity Checking**

In [48]:
import pandas as pd
import csv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files  # For uploading and downloading files in Colab

# Step 1: Upload the CSV file
print("Please upload your dataset (CSV format):")
uploaded = files.upload()

# Get the file name of the uploaded file
file_name = list(uploaded.keys())[0]

# Read the uploaded CSV file
data = []
with open(file_name, mode='r') as file:
    csvFile = csv.reader(file)
    header = next(csvFile)
    for lines in csvFile:
        data.append(lines)

# Convert the data to a DataFrame
df = pd.DataFrame(data, columns=header)

# Step 2: Load the pre-trained Sentence-BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # You can use other models if needed

# Step 3: Generate embeddings for Real Answer and Answer
df['Real Answer Embedding'] = df['Real Answer'].apply(lambda x: model.encode(str(x)))
df['Answer Embedding'] = df['Answer'].apply(lambda x: model.encode(str(x)))

# Step 4: Compute Cosine Similarity Between the Embeddings
similarity_scores = [
    cosine_similarity([df['Real Answer Embedding'][i]], [df['Answer Embedding'][i]])[0][0]
    for i in range(len(df))
]

# Step 5: Normalize the Rating (If rating exists in your dataset)
df['Rating Normalized'] = df['Rating'].astype(float) / 10.0

# Step 6: Adjust Similarity Based on Rating
df['Adjusted Similarity'] = [
    similarity_scores[i] * df['Rating Normalized'][i]
    for i in range(len(df))
]

# Step 7: Save the updated dataset with semantic similarity scores
output_file_name = "AutoGrader_DataSet_with_Semantic_Similarity.csv"
df.to_csv(output_file_name, index=False)

print(f"Updated dataset with adjusted semantic similarity scores saved to: {output_file_name}")

# Step 8: Download the updated dataset
files.download(output_file_name)

Please upload your dataset (CSV format):


Saving processed_TestSet.csv to processed_TestSet (1).csv
Updated dataset with adjusted semantic similarity scores saved to: AutoGrader_DataSet_with_Semantic_Similarity.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>